## 1️⃣ Configuration PySpark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Créer la session Spark
spark = SparkSession.builder \
    .appName("FreshKart-Migration") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"✅ Spark {spark.version} démarré")
print(f"📊 Spark UI: {spark.sparkContext.uiWebUrl}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/20 13:26:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 3.5.0 démarré
📊 Spark UI: http://eb35a4f3dd4f:4040


## 2️⃣ Chargement des données

In [2]:
# Chemins des fichiers (dans le conteneur Docker)
customers_path = "/workspace/Brief_Starter_Pack/data/march-input/customers.csv"
orders_pattern = "/workspace/Brief_Starter_Pack/data/march-input/orders_*.json"
refunds_path = "/workspace/Brief_Starter_Pack/data/march-input/refunds.csv"

start_time = time.time()

# ⚡ PYSPARK : Chargement parallèle intelligent
print("🚀 Chargement des données...")

# Orders : 31 fichiers JSON en une ligne !
orders_df = spark.read \
    .option("multiline", "true") \
    .json(orders_pattern)

# Customers
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(customers_path)

# Refunds
refunds_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(refunds_path)

loading_time = time.time() - start_time
print(f"⚡ Chargement terminé en {loading_time:.2f}s")

# Aperçu
print(f"\n📊 Volumes:")
print(f"- Commandes: {orders_df.count():,}")
print(f"- Clients: {customers_df.count():,}")
print(f"- Remboursements: {refunds_df.count():,}")


🚀 Chargement des données...


⚡ Chargement terminé en 7.35s

📊 Volumes:
- Commandes: 3,193
- Clients: 800
- Remboursements: 1,122


## 3️⃣ Exploration des données

In [3]:
# Schema des commandes
print("📋 Schema des commandes:")
orders_df.printSchema()

# Aperçu des données
print("\n🔍 Aperçu des commandes:")
orders_df.select("order_id", "customer_id", "order_date", "total", "status").show(5)

📋 Schema des commandes:
root
 |-- channel: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment_status: string (nullable = true)


🔍 Aperçu des commandes:


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `order_date` cannot be resolved. Did you mean one of the following? [`order_id`, `created_at`, `items`, `channel`, `customer_id`].;
'Project [order_id#4, customer_id#2, 'order_date, 'total, 'status]
+- Relation [channel#0,created_at#1,customer_id#2,items#3,order_id#4,payment_status#5] json


In [ ]:
# Schema des clients
print("📋 Schema des clients:")
customers_df.printSchema()

print("\n🔍 Aperçu des clients:")
customers_df.show(5)

## 4️⃣ Transformations PySpark

### Explosion des items (nested data)

In [ ]:
# ⚡ PUISSANCE PYSPARK : Gérer les données imbriquées
orders_exploded = orders_df \
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "total",
        "status",
        explode("items").alias("item")
    ) \
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "total",
        "status",
        col("item.product_id").alias("product_id"),
        col("item.quantity").alias("quantity"),
        col("item.price").alias("price")
    )

print("📦 Items explosés:")
orders_exploded.show(10)

### Jointures et agrégations

In [ ]:
# Join orders + customers
orders_with_customers = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left"
)

print("👥 Commandes avec infos clients:")
orders_with_customers.select(
    "order_id",
    orders_df.customer_id,
    "name",
    "email",
    "total"
).show(10)

In [ ]:
# Agrégations par client
customer_stats = orders_df \
    .groupBy("customer_id") \
    .agg(
        count("order_id").alias("nb_orders"),
        sum("total").alias("total_spent"),
        avg("total").alias("avg_order"),
        max("order_date").alias("last_order")
    ) \
    .orderBy(desc("total_spent"))

print("💰 Top 10 clients:")
customer_stats.show(10)

## 5️⃣ Analyse des remboursements

In [ ]:
# Taux de remboursement
refund_rate = refunds_df \
    .groupBy("reason") \
    .agg(
        count("*").alias("count"),
        sum("amount").alias("total_amount")
    ) \
    .orderBy(desc("count"))

print("🔄 Remboursements par raison:")
refund_rate.show()

## 6️⃣ Sauvegarde des résultats

In [ ]:
# Sauvegarder en parquet (format optimisé)
output_path = "/workspace/data/output"

customer_stats \
    .write \
    .mode("overwrite") \
    .parquet(f"{output_path}/customer_stats")

orders_exploded \
    .write \
    .mode("overwrite") \
    .parquet(f"{output_path}/orders_items")

print("✅ Résultats sauvegardés en Parquet !")

## 🎯 Résumé des avantages PySpark

### ✅ Ce qu'on a fait:
- Chargement de 31 fichiers JSON en **1 ligne de code**
- Traitement **lazy** = optimisations automatiques
- **Explosion** de données imbriquées sans boucles
- **Jointures** scalables sur gros volumes
- **Agrégations** distribuées
- Sauvegarde en **Parquet** (format colonnaire optimisé)

### 🚀 Prêt pour la production:
- Ajoutez plus de workers Spark
- Même code fonctionne sur 1TB+ de données
- Resilient Distributed Datasets (RDD) pour fault tolerance

In [ ]:
# Arrêter Spark proprement
spark.stop()
print("👋 Spark arrêté")